### Lab 1.3: Multi-Class Linear Classifier

In this lab you will explore multi-class classification and evaluate model generalization using a [dataset for heart disease prediction from the UCI ML repository](https://archive.ics.uci.edu/dataset/45/heart+disease).

In [ ]:
!pip install -q -r https://raw.githubusercontent.com/calpoly-data4620/DATA-4620-Labs/refs/heads/main/requirements.txt

This ``ucimlrepo`` package provides a nice interface for accessing their datasets.

In [16]:
import numpy as np
from ucimlrepo import fetch_ucirepo 

# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
X = heart_disease.data.features 
y = heart_disease.data.targets 
  
# variable information 
heart_disease.variables


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,NaN,years,no
1,sex,Feature,Categorical,Sex,NaN,NaN,no
2,cp,Feature,Categorical,NaN,NaN,NaN,no
3,trestbps,Feature,Integer,NaN,resting blood pressure (on admission to the ho...,mm Hg,no
4,chol,Feature,Integer,NaN,serum cholestoral,mg/dl,no
5,fbs,Feature,Categorical,NaN,fasting blood sugar > 120 mg/dl,NaN,no
6,restecg,Feature,Categorical,NaN,NaN,NaN,no
7,thalach,Feature,Integer,NaN,maximum heart rate achieved,NaN,no
8,exang,Feature,Categorical,NaN,exercise induced angina,NaN,no
9,oldpeak,Feature,Integer,NaN,ST depression induced by exercise relative to ...,NaN,no


Here I remove the missing values from the features and labels.

In [17]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

Finally I convert the DataFrames to numpy arrays.

In [18]:
X = X.values
y = y.values.flatten()

The classification target is a number from 0-4 indicating the severity of heart disease.  Let's try fitting a linear model.

In [19]:
import sklearn

In [20]:
model = sklearn.linear_model.LogisticRegression().fit(X,y)

/Users/julialu/Documents/calpolyslo/DATA-4620-Labs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
model.score(X,y)

0.6094276094276094

### Exercises

1. Compute the $\mathbf{z}$ values for the classifier manually, i.e. compute

$$\mathbf{z}_i = \mathbf{W}\mathbf{x}_i+\mathbf{b}$$

for each data point $\mathbf{x}_i$.

Stack the resulting $\mathbf{z}_i$ vectors into a $N \times 5$ matrix $\mathbf{Z}$ where $N$ is the number of data points.

Do this first with a `for` loop.  Then do it without a `for` loop, instead using matrix multiplication and broadcasting.

*Hints*: 
- Use `.shape` to get the shape of a Numpy matrix.
- ``@`` is the matrix multiplication operator in Numpy.
- For the version without a for loop, you will need to use the matrix transpose which is `.T` in Numpy.

In [22]:
import numpy as np

W = model.coef_
b = model.intercept_
z = []

# for each data point xi
for x_row in range(X.shape[0]):
        zi = W @ X[x_row] + b
        z.append(zi)
        

z = np.stack(z, axis=0)
z.shape

(297, 5)

In [23]:
W = model.coef_
b = model.intercept_

z = X @ W.T + b
z.shape

(297, 5)

Print out the $\mathbf{z}$ values for the first example in the dataset and the first label.   Determine if the classifier is correctly classifying the first example in the dataset.

In [24]:
z[0]

array([ 1.02965892,  0.44582751, -0.31957004, -0.33765427, -0.81826211])

2. Use ``sklearn.model_selection.train_test_split`` to split ``X`` and ``y`` into 90% train and 10% test splits.  Note that this should be done in a single call to ``train_test_split``.

*Note*: Pass ``random_state=1234`` to ``train_test_split`` to ensure you get the same result from random shuffling each time.


In [25]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.1, random_state=1234)

Fit the model to the training split and calculate accuracy on the test split.  How does it compare to the previous accuracy value (when the model was trained and evaluated on the same data)?

In [26]:
y_train = y_train.flatten()
y_test = y_test.flatten()
model2 = sklearn.linear_model.LogisticRegression().fit(X_train,y_train)
model2.score(X_test, y_test)

/Users/julialu/Documents/calpolyslo/DATA-4620-Labs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.5

The accuracy got worse, from ~60% to 50%. This is because the previous model trained and tested on the same data, so it was overfitting. The new model trained and tested on separate data, which is more realistic and the accuracy went down.

3. Run $k$-fold cross validation with $k=5$ and interpret the results (see `sklearn.model_selection.cross_val_score`).

In [32]:
model_cv = sklearn.linear_model.LogisticRegression()
scores = sklearn.model_selection.cross_val_score(model_cv, X, y, cv=5)
scores

/Users/julialu/Documents/calpolyslo/DATA-4620-Labs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/julialu/Documents/calpolyslo/DATA-4620-Labs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as s

array([0.6       , 0.6       , 0.52542373, 0.55932203, 0.59322034])

In [33]:
scores.mean(), scores.std()

(np.float64(0.5755932203389831), np.float64(0.029270553416844595))

Using 5-fold cross validation, the model got accuracy scores of 0.600, 0.600, 0.525, 0.559, and 0.593. The mean accuracy was about 0.576, or 57.6%, with a standard deviation of about 0.029. This means the model correctly predicts the heart disease severity class a little over half the time on unseen validation folds.

The cross-validation accuracy is lower than the original 60.9% accuracy from training and testing on the same full dataset, which makes sense because cross validation evaluates the model on data it did not train on. It is slightly better than the single train/test split accuracy of 50%. Cross validation gives a more reliable estimate of generalization because it averages performance across 5 different splits.